# sequence_trend_joint_bc_screen_v1

Notebook-first diagnostic screen for the current model-selection bottleneck.

- **Baselines**: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct comparison `trend_participation_bc_warmstart_screen_v1` plus `cap10_sequence_risk_target_cap_bc_screen_v1`.
- **Role**: diagnose, not promote.
- **Changed layer**: training target composition and BC warm start only.
- **External basis**: Qlib/FinRL walk-forward discipline, behavioral cloning/DAgger-style supervised warm starts, and drawdown-controlled momentum literature listed in `selection.json`.
- **Hypothesis**: prior trend targets fix 2023-2024 participation but blow up 2022, while sequence/drawdown targets reduce 2022 risk but lose 2023-2024. A risk-priority joint contract should preserve sequence caps on risk dates and only overlay trend participation on non-risk dates.
- **Acceptance criteria**: a non-baseline candidate passes `relative_return >= +30pp`, `MaxDD <= 15%`, slice fit, and drift gate on both `wf_2022` and `wf_2023_2024`. Passing here only permits a later PPO/checkpoint screen; promotion remains blocked.


In [1]:

from __future__ import annotations

import json
import math
import os
import random
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
os.chdir(ROOT)
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.dashboard import write_review_daily_csv
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
    score_bc_walkforward_drift,
    summarize_bc_walkforward_drift_screen,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.policy.policy_builder import state_objective_target_position_from_row
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = "sequence_trend_joint_bc_screen_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
TREND_TARGETS_PATH = ROOT / "runtime/model_selection/card/trend_participation_foldlocal_target_v1/selected_trend_foldlocal_targets.csv"
BASE_TARGETS_PATH = ROOT / "runtime/model_selection/card/cap10_sequence_risk_target_cap_bc_screen_v1/baseline_no_sequence_cap_target_contract.csv"
SLOPE_LOW_TARGETS_PATH = ROOT / "runtime/model_selection/card/cap10_sequence_risk_target_cap_bc_screen_v1/slope_low_cap15_target_contract.csv"
TREND_BREAK_TARGETS_PATH = ROOT / "runtime/model_selection/card/cap10_sequence_risk_target_cap_bc_screen_v1/trend_break_any_cap10_target_contract.csv"
TREND_SOURCE_SELECTION_PATH = ROOT / "runtime/model_selection/card/trend_participation_bc_warmstart_screen_v1/selection.json"
SEQUENCE_SOURCE_SELECTION_PATH = ROOT / "runtime/model_selection/card/cap10_sequence_risk_target_cap_bc_screen_v1/selection.json"
FRAME_PATH = ROOT / "data/soxl_profile_balanced_rule_residual_v22.pkl"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
DRIFT_RELATIVE_FLOOR = -5.0
DRIFT_MAX_DD_CEILING = 2.0
GLOBAL_ACCURACY_MIN = 0.70
SPARSE_SLICE_ACCURACY_MIN = 0.60
RISK_SLICE_ACCURACY_MIN = 0.70
TREND_SLICE_MIN_SAMPLES = 5
RISK_SLICE_MIN_SAMPLES = 50
BC_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)
WALK_FORWARD_WINDOWS = {
    "wf_2022": {"target_fold": "wf_2022_train", "train": ("2010-03-11", "2020-12-31"), "valid": ("2021-01-01", "2021-12-31"), "test": ("2022-01-01", "2022-12-31")},
    "wf_2023_2024": {"target_fold": "wf_2023_2024_train", "train": ("2010-03-11", "2022-12-31"), "valid": ("2023-01-01", "2023-12-31"), "test": ("2024-01-01", "2024-12-31")},
}
EXTERNAL_SOURCES = [
    "https://github.com/microsoft/qlib",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://proceedings.mlr.press/v15/ross11a.html",
    "https://arxiv.org/abs/1904.04912",
    "https://doi.org/10.21314/JCF.2016.322",
]

def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.floating):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value

def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_targets(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, parse_dates=["datetime"])
    for col in ["is_rebound_cluster", "is_downtrend_state", "release_gate_state", "drawdown_control_state", "sequence_risk_state", "trend_participation_signal_ready", "trend_participation_guard_ready", "trend_participation_target_state", "trend_participation_conflict_state"]:
        if col not in frame.columns:
            frame[col] = False
        frame[col] = frame[col].fillna(False).astype(bool)
    for col in ["teacher_position", "rebound_target_position", "risk_target_position", "objective_weight"]:
        frame[col] = pd.to_numeric(frame[col], errors="coerce").fillna(0.0)
    if "risk_target_position_original" not in frame.columns:
        frame["risk_target_position_original"] = frame["risk_target_position"]
    if "target_recipe" not in frame.columns:
        frame["target_recipe"] = frame.get("candidate", "")
    if "source_components" not in frame.columns:
        frame["source_components"] = frame.get("candidate", "")
    frame["effective_target_position"] = [state_objective_target_position_from_row(row) for row in frame.to_dict("records")]
    return frame.sort_values(["fold", "datetime"]).reset_index(drop=True)

def risk_block_mask(frame: pd.DataFrame) -> pd.Series:
    mask = pd.Series(False, index=frame.index)
    for col in ["sequence_risk_state", "drawdown_control_state", "is_downtrend_state"]:
        if col in frame.columns:
            mask = mask | frame[col].fillna(False).astype(bool)
    return mask

def overlay_trend_on_nonrisk_rows(risk_targets: pd.DataFrame, trend_targets: pd.DataFrame, *, candidate_name: str) -> pd.DataFrame:
    base = risk_targets.copy()
    trend = trend_targets.copy()
    union_cols = sorted(set(base.columns).union(set(trend.columns)).union({"joint_overlay_policy", "trend_participation_conflict_state", "joint_candidate"}))
    for col in union_cols:
        if col not in base.columns:
            base[col] = False if col.endswith("_state") or col.startswith("is_") else ""
        if col not in trend.columns:
            trend[col] = False if col.endswith("_state") or col.startswith("is_") else ""
    base = base[union_cols].copy()
    trend = trend[union_cols].copy()
    base["datetime"] = pd.to_datetime(base["datetime"]).dt.normalize()
    trend["datetime"] = pd.to_datetime(trend["datetime"]).dt.normalize()
    base["joint_overlay_policy"] = "risk_contract_row"
    base["trend_participation_conflict_state"] = False
    base["joint_candidate"] = candidate_name
    trend_rows = trend[trend["trend_participation_target_state"].fillna(False).astype(bool)].copy()
    base_index = {(str(row["fold"]), pd.Timestamp(row["datetime"]).normalize()): idx for idx, row in base.iterrows()}
    appended = []
    replaced = 0
    conflicts = 0
    for _, trow in trend_rows.iterrows():
        key = (str(trow["fold"]), pd.Timestamp(trow["datetime"]).normalize())
        idx = base_index.get(key)
        if idx is None:
            new_row = trow.copy()
            new_row["candidate"] = candidate_name
            new_row["target_recipe"] = candidate_name
            new_row["joint_overlay_policy"] = "trend_appended"
            new_row["joint_candidate"] = candidate_name
            appended.append(new_row)
            continue
        if bool(risk_block_mask(base.loc[[idx]]).iloc[0]):
            base.at[idx, "trend_participation_conflict_state"] = True
            base.at[idx, "joint_overlay_policy"] = "risk_priority_trend_conflict"
            conflicts += 1
            continue
        for col in union_cols:
            if col in {"joint_overlay_policy", "trend_participation_conflict_state", "joint_candidate"}:
                continue
            base.at[idx, col] = trow[col]
        base.at[idx, "candidate"] = candidate_name
        base.at[idx, "target_recipe"] = candidate_name
        base.at[idx, "joint_overlay_policy"] = "trend_replaced_nonrisk"
        base.at[idx, "joint_candidate"] = candidate_name
        replaced += 1
    if appended:
        base = pd.concat([base, pd.DataFrame(appended)], ignore_index=True)
    base["candidate"] = candidate_name
    base["target_recipe"] = candidate_name
    for col in ["is_rebound_cluster", "is_downtrend_state", "release_gate_state", "drawdown_control_state", "sequence_risk_state", "trend_participation_target_state", "trend_participation_conflict_state"]:
        if col in base.columns:
            base[col] = base[col].fillna(False).astype(bool)
    base["effective_target_position"] = [state_objective_target_position_from_row(row) for row in base.to_dict("records")]
    base.attrs["trend_replaced_rows"] = int(replaced)
    base.attrs["trend_conflict_rows"] = int(conflicts)
    base.attrs["trend_appended_rows"] = int(len(appended))
    return base.sort_values(["fold", "datetime"]).reset_index(drop=True)

def fold_target_subset(targets: pd.DataFrame, spec: dict[str, Any]) -> pd.DataFrame:
    train_end = pd.Timestamp(spec["train"][1])
    subset = targets[(targets["fold"].astype(str).eq(str(spec["target_fold"]))) & (pd.to_datetime(targets["datetime"]) <= train_end)].copy()
    return subset.sort_values("datetime").reset_index(drop=True)

def price_slice(dataset, segment: tuple[str, str]) -> pd.Series:
    return dataset.handler.price_frame.droplevel("instrument")["price"].sort_index().loc[segment[0]:segment[1]]

def evaluate_actor_path(actor, eval_df: pd.DataFrame, price_series: pd.Series, profile, test_seg: tuple[str, str], *, label: str) -> tuple[dict[str, float], Path]:
    actor.eval()
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(actor, eval_df, position_cap_config=profile.position_cap, position_budget_config=profile.position_budget, execution_config=profile.execution, asset_idx=0, action_levels=action_levels_for_profile(profile), observation_window=profile.model.observation_window, hard_floor=profile.model.hard_floor, use_hybrid_overlay=profile.model.use_hybrid_overlay, delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None, feature_columns=feature_columns, mode_router_config=profile.mode_router, position_mapping=profile.model.position_mapping)
    bt = backtest_profile(actor, eval_df, price_series, position_cap_config=profile.position_cap, position_budget_config=profile.position_budget, execution_config=profile.execution, asset_idx=0, feature_columns=feature_columns, model_config=profile.model, mode_router_config=profile.mode_router, test_seg=test_seg)
    out = dict(split_metrics)
    out["backtest_return"] = float(bt.get("return", out.get("return", 0.0)))
    out["backtest_max_dd"] = float(bt.get("max_dd", out.get("max_dd", 0.0)))
    out["backtest_sharpe"] = float(bt.get("sharpe", out.get("sharpe", 0.0)))
    daily_path = CARD / f"{label}_daily_path.csv"
    write_review_daily_csv(daily_path, bt["portfolio"], price_series, bt.get("position_history", []), bt.get("trade_signals", []))
    return {k: float(v) for k, v in out.items() if isinstance(v, (int, float, np.integer, np.floating))}, daily_path

def slice_row(metrics: pd.DataFrame, name: str):
    if metrics.empty or name not in set(metrics["slice"].astype(str)):
        return None
    return metrics[metrics["slice"].astype(str).eq(name)].iloc[0]

def fit_slice_gate(slice_metrics: pd.DataFrame, *, trend_samples: int, sequence_samples: int, drawdown_samples: int) -> tuple[bool, dict[str, Any]]:
    failed = []
    details: dict[str, Any] = {}
    row = slice_row(slice_metrics, "all")
    if row is None:
        failed.append("all_slice_missing")
    else:
        details["all_accuracy"] = float(row["accuracy"])
        if not math.isfinite(float(row["accuracy"])) or float(row["accuracy"]) < GLOBAL_ACCURACY_MIN:
            failed.append("all_accuracy")
    if trend_samples >= TREND_SLICE_MIN_SAMPLES:
        row = slice_row(slice_metrics, "trend_participation_target_state")
        if row is None:
            failed.append("trend_slice_missing")
            details["trend_accuracy"] = None
        else:
            details["trend_samples"] = int(row["samples"])
            details["trend_accuracy"] = float(row["accuracy"])
            if int(row["samples"]) < TREND_SLICE_MIN_SAMPLES or float(row["accuracy"]) < SPARSE_SLICE_ACCURACY_MIN:
                failed.append("trend_slice_accuracy")
    else:
        details["trend_samples"] = int(trend_samples)
        details["trend_accuracy"] = None
    for slice_name, sample_count in [("sequence_risk_state", sequence_samples), ("drawdown_control_state", drawdown_samples)]:
        details[f"{slice_name}_samples"] = int(sample_count)
        if sample_count < RISK_SLICE_MIN_SAMPLES:
            details[f"{slice_name}_accuracy"] = None
            continue
        row = slice_row(slice_metrics, slice_name)
        if row is None:
            failed.append(f"{slice_name}_missing")
            details[f"{slice_name}_accuracy"] = None
            continue
        details[f"{slice_name}_accuracy"] = float(row["accuracy"])
        if int(row["samples"]) < RISK_SLICE_MIN_SAMPLES or float(row["accuracy"]) < RISK_SLICE_ACCURACY_MIN:
            failed.append(f"{slice_name}_accuracy")
    details["failed_slice_checks"] = failed
    return len(failed) == 0, details

def metric_loss(row: pd.Series) -> float:
    rel = float(row.get("candidate_relative_return", float("nan")))
    dd = float(row.get("candidate_max_dd", float("nan")))
    rel_delta = float(row.get("relative_delta_vs_baseline", float("nan")))
    dd_delta = float(row.get("max_dd_delta_vs_baseline", float("nan")))
    loss = 0.0
    loss += max(0.0, TARGET_RELATIVE_RETURN - rel) / 30.0 if math.isfinite(rel) else 10.0
    loss += max(0.0, dd - TARGET_MAX_DD) / 15.0 if math.isfinite(dd) else 10.0
    loss += max(0.0, DRIFT_RELATIVE_FLOOR - rel_delta) / 10.0 if math.isfinite(rel_delta) else 5.0
    loss += max(0.0, dd_delta - DRIFT_MAX_DD_CEILING) / 5.0 if math.isfinite(dd_delta) else 5.0
    if not bool(row.get("slice_gate", False)):
        loss += 1.0
    return float(loss)

trend_source_selection = json.loads(TREND_SOURCE_SELECTION_PATH.read_text(encoding="utf-8"))
sequence_source_selection = json.loads(SEQUENCE_SOURCE_SELECTION_PATH.read_text(encoding="utf-8"))
profile = load_profile(PROFILE_NAME)
frame = pd.read_pickle(FRAME_PATH)
base_targets = load_targets(BASE_TARGETS_PATH)
trend_targets = load_targets(TREND_TARGETS_PATH)
slope_targets = load_targets(SLOPE_LOW_TARGETS_PATH)
trend_break_targets = load_targets(TREND_BREAK_TARGETS_PATH)
joint_slope_targets = overlay_trend_on_nonrisk_rows(slope_targets, trend_targets, candidate_name="trend_plus_slope_low_cap15")
joint_break_targets = overlay_trend_on_nonrisk_rows(trend_break_targets, trend_targets, candidate_name="trend_plus_trend_break_any_cap10")
for name, targets in {"trend_plus_slope_low_cap15": joint_slope_targets, "trend_plus_trend_break_any_cap10": joint_break_targets}.items():
    targets.to_csv(CARD / f"{name}_target_contract.csv", index=False)
targets_by_source = {"base": base_targets, "trend": trend_targets, "joint_slope": joint_slope_targets, "joint_break": joint_break_targets}
CANDIDATES = [
    {"candidate": "base_plain_contract", "target_source": "base", "complexity": 0, "sample_weight_config": None, "description": "Unchanged fold-local contract; drift baseline."},
    {"candidate": "trend_plain_contract", "target_source": "trend", "complexity": 1, "sample_weight_config": None, "description": "Previous trend participation target; direct comparison."},
    {"candidate": "trend_plus_slope_low_cap15", "target_source": "joint_slope", "complexity": 2, "sample_weight_config": None, "description": "Risk-priority slope-low cap15 contract plus non-risk trend participation overlay."},
    {"candidate": "trend_plus_slope_low_cap15_weighted", "target_source": "joint_slope", "complexity": 3, "sample_weight_config": StateObjectiveBCSampleWeightConfig(rebound_multiplier=3.0, downtrend_multiplier=2.0, ordinary_multiplier=1.0, objective_weight_power=0.20, class_balanced_beta=0.99, weight_cap=25.0), "description": "Same joint contract with moderate risk/trend slice and class-balance weighting."},
    {"candidate": "trend_plus_trend_break_any_cap10", "target_source": "joint_break", "complexity": 4, "sample_weight_config": None, "description": "Trend-break cap10 risk contract plus non-risk trend overlay, preserving cap10 on conflicts."},
]
BASELINE_CANDIDATE = "base_plain_contract"
config_rows = []
coverage_rows = []
for item in CANDIDATES:
    cfg = item["sample_weight_config"]
    targets = targets_by_source[item["target_source"]]
    config_rows.append({"candidate": item["candidate"], "target_source": item["target_source"], "complexity": int(item["complexity"]), "sample_weight_config": None if cfg is None else asdict(cfg), "description": item["description"]})
    for fold_name, spec in WALK_FORWARD_WINDOWS.items():
        subset = fold_target_subset(targets, spec)
        coverage_rows.append({"candidate": item["candidate"], "fold": fold_name, "target_fold": spec["target_fold"], "target_rows": int(len(subset)), "unique_target_dates": int(subset["datetime"].nunique()) if len(subset) else 0, "trend_target_rows": int(subset["trend_participation_target_state"].sum()) if len(subset) else 0, "trend_conflict_rows": int(subset["trend_participation_conflict_state"].sum()) if len(subset) else 0, "sequence_risk_rows": int(subset["sequence_risk_state"].sum()) if len(subset) else 0, "drawdown_control_rows": int(subset["drawdown_control_state"].sum()) if len(subset) else 0, "rebound_rows": int(subset["is_rebound_cluster"].sum()) if len(subset) else 0, "downtrend_rows": int(subset["is_downtrend_state"].sum()) if len(subset) else 0, "min_effective_target": float(subset["effective_target_position"].min()) if len(subset) else np.nan, "max_effective_target": float(subset["effective_target_position"].max()) if len(subset) else np.nan, "train_start": spec["train"][0], "train_end": spec["train"][1], "valid_start": spec["valid"][0], "valid_end": spec["valid"][1], "test_start": spec["test"][0], "test_end": spec["test"][1]})
candidate_configs = pd.DataFrame(config_rows)
coverage = pd.DataFrame(coverage_rows)
candidate_configs.to_csv(CARD / "bc_candidate_configs.csv", index=False)
coverage.to_csv(CARD / "coverage.csv", index=False)
write_json(CARD / "coverage.json", {"experiment": EXPERIMENT, "profile": PROFILE_NAME, "frame_path": str(FRAME_PATH.relative_to(ROOT)), "baseline_targets": str(BASE_TARGETS_PATH.relative_to(ROOT)), "trend_targets": str(TREND_TARGETS_PATH.relative_to(ROOT)), "slope_low_targets": str(SLOPE_LOW_TARGETS_PATH.relative_to(ROOT)), "trend_break_targets": str(TREND_BREAK_TARGETS_PATH.relative_to(ROOT)), "joint_target_contracts": ["trend_plus_slope_low_cap15_target_contract.csv", "trend_plus_trend_break_any_cap10_target_contract.csv"], "windows": coverage_rows, "skipped_windows": [{"fold": "wf_2020", "reason": "target rows start in 2020/2021; using them would leak the wf_2020 test window"}]})
display(candidate_configs)
display(coverage)

init_qlib(QLIB_DATA_PATH)
fold_dataset_cache = {}
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(train_seg=spec["train"], valid_seg=spec["valid"], test_seg=spec["test"], feature_packs=tuple(profile.feature.packs), profile_name=profile.name)
    fold_dataset_cache[fold_name] = {"dataset": dataset, "valid_df": segment_frame(dataset, "valid"), "test_df": segment_frame(dataset, "test"), "valid_price": price_slice(dataset, spec["valid"]), "test_price": price_slice(dataset, spec["test"])}
fold_rows = []
slice_rows = []
action_rows = []
daily_rows = []
for item in CANDIDATES:
    candidate = item["candidate"]
    set_seed(SEED + int(item["complexity"]))
    targets = targets_by_source[item["target_source"]]
    for fold_name, spec in WALK_FORWARD_WINDOWS.items():
        fold_targets = fold_target_subset(targets, spec)
        if fold_targets.empty:
            fold_rows.append({"candidate": candidate, "fold": fold_name, "status": "skipped_no_targets"})
            continue
        fit = fit_state_objective_bc_actor_from_frame(frame, targets=fold_targets, profile=profile, train_config=BC_TRAIN_CONFIG, sample_weight_config=item["sample_weight_config"], sample_size=0, rng_seed=SEED)
        audit = build_state_objective_bc_dataset_from_frame(frame, targets=fold_targets, profile=profile, sample_size=0, rng_seed=SEED)
        slice_metrics = evaluate_bc_actor_slice_fit(fit.actor, fit.obs, fit.actions, audit.metadata, slice_cols=("trend_participation_target_state", "is_rebound_cluster", "is_downtrend_state", "release_gate_state", "drawdown_control_state", "sequence_risk_state", "trend_participation_conflict_state"))
        meta = audit.metadata
        trend_samples = int(meta["trend_participation_target_state"].fillna(False).astype(bool).sum()) if "trend_participation_target_state" in meta else 0
        sequence_samples = int(meta["sequence_risk_state"].fillna(False).astype(bool).sum()) if "sequence_risk_state" in meta else 0
        drawdown_samples = int(meta["drawdown_control_state"].fillna(False).astype(bool).sum()) if "drawdown_control_state" in meta else 0
        slice_gate, slice_summary = fit_slice_gate(slice_metrics, trend_samples=trend_samples, sequence_samples=sequence_samples, drawdown_samples=drawdown_samples)
        tagged = slice_metrics.copy(); tagged.insert(0, "candidate", candidate); tagged.insert(1, "fold", fold_name); slice_rows.extend(tagged.to_dict("records"))
        dist = fit.action_distribution.copy(); dist.insert(0, "candidate", candidate); dist.insert(1, "fold", fold_name); action_rows.extend(dist.to_dict("records"))
        cache = fold_dataset_cache[fold_name]
        valid_metrics, valid_daily = evaluate_actor_path(fit.actor, cache["valid_df"], cache["valid_price"], profile, spec["valid"], label=f"{candidate}_{fold_name}_valid")
        test_metrics, test_daily = evaluate_actor_path(fit.actor, cache["test_df"], cache["test_price"], profile, spec["test"], label=f"{candidate}_{fold_name}_test")
        daily_rows.extend([{"candidate": candidate, "fold": fold_name, "segment": "valid", "path": str(valid_daily.relative_to(ROOT))}, {"candidate": candidate, "fold": fold_name, "segment": "test", "path": str(test_daily.relative_to(ROOT))}])
        fold_rows.append({"candidate": candidate, "fold": fold_name, "status": "evaluated", "target_rows": int(len(fold_targets)), "bc_dataset_samples": int(len(audit.actions)), "trend_target_samples": int(trend_samples), "sequence_risk_samples": int(sequence_samples), "drawdown_control_samples": int(drawdown_samples), "after_accuracy": float(fit.metrics["after_accuracy"]), "after_loss": float(fit.metrics["after_loss"]), "slice_gate": bool(slice_gate), "slice_failed_checks": json.dumps(slice_summary["failed_slice_checks"], sort_keys=True), "all_accuracy": slice_summary.get("all_accuracy"), "trend_accuracy": slice_summary.get("trend_accuracy"), "sequence_risk_accuracy": slice_summary.get("sequence_risk_state_accuracy"), "drawdown_control_accuracy": slice_summary.get("drawdown_control_state_accuracy"), "valid_relative_return": float(valid_metrics.get("relative_return", float("nan"))), "valid_max_dd": float(valid_metrics.get("max_dd", float("nan"))), "valid_avg_pos": float(valid_metrics.get("avg_pos", float("nan"))), "test_relative_return": float(test_metrics.get("relative_return", float("nan"))), "test_max_dd": float(test_metrics.get("max_dd", float("nan"))), "test_avg_pos": float(test_metrics.get("avg_pos", float("nan"))), "test_return": float(test_metrics.get("return", float("nan"))), "test_sharpe": float(test_metrics.get("sharpe", float("nan")))})
fold_results = pd.DataFrame(fold_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
slice_metrics = pd.DataFrame(slice_rows).sort_values(["candidate", "fold", "slice"]).reset_index(drop=True)
action_distribution = pd.DataFrame(action_rows).sort_values(["candidate", "fold", "action"]).reset_index(drop=True)
daily_manifest = pd.DataFrame(daily_rows).sort_values(["candidate", "fold", "segment"]).reset_index(drop=True)
fold_results.to_csv(CARD / "bc_fold_results.csv", index=False)
slice_metrics.to_csv(CARD / "bc_slice_metrics.csv", index=False)
action_distribution.to_csv(CARD / "bc_action_distribution.csv", index=False)
daily_manifest.to_csv(CARD / "bc_daily_manifest.csv", index=False)
display(fold_results)

evaluated = fold_results[fold_results["status"].eq("evaluated")].copy()
baseline_by_fold = {row["fold"]: row for row in evaluated[evaluated["candidate"].eq(BASELINE_CANDIDATE)].to_dict("records")}
score_rows = []
for row in evaluated.to_dict("records"):
    baseline = baseline_by_fold.get(row["fold"])
    if baseline is None:
        continue
    scored = score_bc_walkforward_drift(str(row["fold"]), candidate_metrics={"relative_return": row["test_relative_return"], "max_dd": row["test_max_dd"], "avg_pos": row["test_avg_pos"]}, baseline_metrics={"relative_return": baseline["test_relative_return"], "max_dd": baseline["test_max_dd"], "avg_pos": baseline["test_avg_pos"]}, slice_gate=bool(row["slice_gate"]), target_relative_return_pp=TARGET_RELATIVE_RETURN, target_max_dd_pct=TARGET_MAX_DD, relative_drift_floor_pp=DRIFT_RELATIVE_FLOOR, max_dd_drift_ceiling_pp=DRIFT_MAX_DD_CEILING)
    scored.update({"candidate": row["candidate"], "status": row["status"], "valid_relative_return": row["valid_relative_return"], "valid_max_dd": row["valid_max_dd"], "valid_avg_pos": row["valid_avg_pos"], "trend_target_samples": int(row["trend_target_samples"]), "sequence_risk_samples": int(row["sequence_risk_samples"]), "drawdown_control_samples": int(row["drawdown_control_samples"]), "after_accuracy": float(row["after_accuracy"]), "slice_failed_checks": row["slice_failed_checks"], "failed_checks_json": json.dumps(scored["failed_checks"], sort_keys=True)})
    score_rows.append(scored)
drift_scores = pd.DataFrame(score_rows)
if not drift_scores.empty:
    drift_scores["gate_loss"] = drift_scores.apply(metric_loss, axis=1)
    drift_scores = drift_scores.sort_values(["candidate", "fold"]).reset_index(drop=True)
drift_scores.to_csv(CARD / "bc_drift_scores.csv", index=False)
summary_rows = []
required_folds = tuple(WALK_FORWARD_WINDOWS.keys())
for candidate, group in drift_scores.groupby("candidate", sort=True):
    summary = summarize_bc_walkforward_drift_screen(group, required_folds=required_folds)
    summary_rows.append({"candidate": candidate, "decision": summary["decision"], "fold_gate_pass_count": summary["fold_gate_pass_count"], "candidate_gate_pass_count": summary["candidate_gate_pass_count"], "baseline_gate_pass_count": summary["baseline_gate_pass_count"], "drift_gate_pass_count": summary["drift_gate_pass_count"], "slice_gate_pass_count": summary["slice_gate_pass_count"], "failed_folds_json": json.dumps(summary["failed_folds"], sort_keys=True), "gate_loss": float(group["gate_loss"].sum()) if "gate_loss" in group else float("inf"), "min_relative_return": float(group["candidate_relative_return"].min()), "max_max_dd": float(group["candidate_max_dd"].max()), "mean_relative_delta_vs_baseline": float(group["relative_delta_vs_baseline"].mean()), "max_dd_delta_vs_baseline": float(group["max_dd_delta_vs_baseline"].max()), "promotion_allowed": False, "next_mechanism": summary["next_mechanism"]})
candidate_screen = pd.DataFrame(summary_rows).sort_values(["fold_gate_pass_count", "candidate_gate_pass_count", "drift_gate_pass_count", "gate_loss"], ascending=[False, False, False, True]).reset_index(drop=True)
candidate_screen.to_csv(CARD / "bc_candidate_screen.csv", index=False)
display(drift_scores)
display(candidate_screen)
non_baseline = candidate_screen[~candidate_screen["candidate"].eq(BASELINE_CANDIDATE)].copy()
selected = non_baseline.iloc[0].to_dict() if not non_baseline.empty else (candidate_screen.iloc[0].to_dict() if not candidate_screen.empty else {})
selected_candidate = str(selected.get("candidate", ""))
selected_scores = drift_scores[drift_scores["candidate"].eq(selected_candidate)].copy() if selected_candidate else pd.DataFrame()
selected_summary = summarize_bc_walkforward_drift_screen(selected_scores, required_folds=tuple(WALK_FORWARD_WINDOWS.keys())) if not selected_scores.empty else {"decision": "bc_walkforward_drift_no_rows", "promotion_allowed": False, "next_mechanism": "repair_bc_screen_before_training"}
all_selected_folds_pass = bool(selected_summary.get("decision") == "bc_walkforward_drift_gate_passed_no_promotion")
any_selected_fold_pass = int(selected_summary.get("fold_gate_pass_count", 0)) > 0
any_candidate_gate = int(selected_summary.get("candidate_gate_pass_count", 0)) > 0
if all_selected_folds_pass:
    decision = "sequence_trend_joint_bc_passed_no_promotion"; next_mechanism = "run_short_ppo_or_checkpoint_screen_with_drawdown_drift_stop_then_fixed_split_wf2020"
elif any_selected_fold_pass:
    decision = "sequence_trend_joint_bc_mixed_no_promotion"; next_mechanism = "inspect_failed_fold_action_distribution_then_try_drawdown_drift_reward_or_checkpoint_selector"
elif any_candidate_gate:
    decision = "sequence_trend_joint_bc_drawdown_or_slice_failed_no_promotion"; next_mechanism = "block_ppo_until_joint_target_drawdown_drift_or_slice_fit_is_fixed"
else:
    decision = "sequence_trend_joint_bc_failed_no_promotion"; next_mechanism = "return_to_sequence_risk_state_or_reward_drawdown_drift_penalty_before_ppo"
selection = {"experiment": EXPERIMENT, "created_at": datetime.now(timezone.utc).isoformat(), "decision": decision, "selected_candidate": selected_candidate, "selected_summary": selected, "selected_walkforward_summary": selected_summary, "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1", "direct_comparison_baseline": ["trend_participation_bc_warmstart_screen_v1", "cap10_sequence_risk_target_cap_bc_screen_v1", "base_plain_contract"], "closest_historical_success_case": "trend_participation_bc_warmstart_screen_v1 fixed wf_2023_2024 but failed wf_2022 drawdown", "closest_historical_failure_case": "cap10_sequence_risk_target_cap_bc_screen_v1 reduced/selected risk states but failed either wf_2022 MaxDD or wf_2023_2024 MaxDD", "why_not_duplicate": "This notebook composes the two mechanisms with risk-date priority and trend overlay only on non-risk dates, then screens BC drift across both folds.", "role": "diagnose", "profile": PROFILE_NAME, "changed_layer": ["model_selection", "training_target_composition", "bc_warm_start"], "target_relative_return_pp": TARGET_RELATIVE_RETURN, "target_max_dd_pct": TARGET_MAX_DD, "drift_relative_floor_pp": DRIFT_RELATIVE_FLOOR, "drift_max_dd_ceiling_pp": DRIFT_MAX_DD_CEILING, "folds_evaluated": list(WALK_FORWARD_WINDOWS.keys()), "wf_2020_coverage": False, "wf_2020_skip_reason": "target rows start in 2020/2021; using them would leak wf_2020 labels", "promotion_allowed": False, "promotion_blocker": "Diagnostic BC warm-start screen only. Promotion requires fixed-split plus wf_2020/wf_2022/wf_2023_2024 replay and retained actor review.", "ppo_screen_allowed_next": bool(all_selected_folds_pass), "training_allowed_from_this_card": bool(all_selected_folds_pass), "actor_retention": "no actor bundle retained; rerun notebook to regenerate diagnostic BC actors", "next_mechanism": next_mechanism, "source_selection": {"trend_bc": trend_source_selection, "sequence_target": sequence_source_selection}, "external_sources": EXTERNAL_SOURCES, "sidecars": ["trend_plus_slope_low_cap15_target_contract.csv", "trend_plus_trend_break_any_cap10_target_contract.csv", "bc_candidate_configs.csv", "coverage.csv", "coverage.json", "bc_fold_results.csv", "bc_slice_metrics.csv", "bc_action_distribution.csv", "bc_daily_manifest.csv", "bc_drift_scores.csv", "bc_candidate_screen.csv", "*_daily_path.csv", "selection.json", "summary.json"]}
summary = {"experiment": EXPERIMENT, "decision": decision, "selected_candidate": selected_candidate, "promotion_allowed": False, "ppo_screen_allowed_next": bool(all_selected_folds_pass), "candidate_screen": candidate_screen.to_dict("records"), "selected_scores": selected_scores.to_dict("records"), "conclusion": [], "next_mechanism": next_mechanism}
if selected_scores.empty:
    summary["conclusion"].append("No selected walk-forward rows were available; do not train further from this card.")
else:
    for row in selected_scores.sort_values("fold").to_dict("records"):
        summary["conclusion"].append(f"{row['fold']}: {selected_candidate} relative={row['candidate_relative_return']:.2f}pp, MaxDD={row['candidate_max_dd']:.2f}%, delta vs base rel={row['relative_delta_vs_baseline']:.2f}pp, delta MaxDD={row['max_dd_delta_vs_baseline']:.2f}pp, slice_gate={bool(row['slice_gate'])}, fold_gate={bool(row['fold_gate'])}.")
    summary["conclusion"].append("No runtime promotion: this card evaluates target composition and BC warm-start behavior only and skips wf_2020 by leakage guard.")
write_json(CARD / "selection.json", selection)
write_json(CARD / "summary.json", summary)
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


,candidate,target_source,complexity,sample_weight_config,description
0,base_plain_contract,base,0,None,Unchanged fold-local contract; drift baseline.
1,trend_plain_contract,trend,1,None,Previous trend participation target; direct co...
2,trend_plus_slope_low_cap15,joint_slope,2,None,Risk-priority slope-low cap15 contract plus no...
3,trend_plus_slope_low_cap15_weighted,joint_slope,3,"{'rebound_multiplier': 3.0, 'downtrend_multipl...",Same joint contract with moderate risk/trend s...
4,trend_plus_trend_break_any_cap10,joint_break,4,None,Trend-break cap10 risk contract plus non-risk ...


,candidate,fold,target_fold,target_rows,unique_target_dates,trend_target_rows,trend_conflict_rows,sequence_risk_rows,drawdown_control_rows,rebound_rows,downtrend_rows,min_effective_target,max_effective_target,train_start,train_end,valid_start,valid_end,test_start,test_end
0,base_plain_contract,wf_2022,wf_2022_train,253,253,0,0,0,49,66,118,0.0625,0.893943,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31
1,base_plain_contract,wf_2023_2024,wf_2023_2024_train,756,756,0,0,0,49,66,478,0.0625,0.893943,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31
2,trend_plain_contract,wf_2022,wf_2022_train,253,253,0,0,0,49,66,118,0.0625,0.893943,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31
3,trend_plain_contract,wf_2023_2024,wf_2023_2024_train,756,756,15,0,0,49,81,478,0.0625,0.893943,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31
4,trend_plus_slope_low_cap15,wf_2022,wf_2022_train,253,253,0,0,64,77,66,128,0.0625,0.893943,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31
5,trend_plus_slope_low_cap15,wf_2023_2024,wf_2023_2024_train,756,756,15,0,189,221,81,483,0.0625,0.893943,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31
6,trend_plus_slope_low_cap15_weighted,wf_2022,wf_2022_train,253,253,0,0,64,77,66,128,0.0625,0.893943,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31
7,trend_plus_slope_low_cap15_weighted,wf_2023_2024,wf_2023_2024_train,756,756,15,0,189,221,81,483,0.0625,0.893943,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31
8,trend_plus_trend_break_any_cap10,wf_2022,wf_2022_train,253,253,0,0,144,144,66,184,0.0625,0.893943,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31
9,trend_plus_trend_break_any_cap10,wf_2023_2024,wf_2023_2024_train,756,756,8,7,401,411,74,630,0.0625,0.825000,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31


[39747:MainThread](2026-05-26 14:51:46,136) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[39747:MainThread](2026-05-26 14:51:47,030) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,candidate,fold,status,target_rows,bc_dataset_samples,trend_target_samples,sequence_risk_samples,drawdown_control_samples,after_accuracy,after_loss,...,sequence_risk_accuracy,drawdown_control_accuracy,valid_relative_return,valid_max_dd,valid_avg_pos,test_relative_return,test_max_dd,test_avg_pos,test_return,test_sharpe
0,base_plain_contract,wf_2022,evaluated,253,253,0,0,49,0.833992,1.073453,...,NaN,NaN,-58.602980,12.185863,0.330955,36.419124,43.571257,0.280667,-40.195132,-1.551696
1,base_plain_contract,wf_2023_2024,evaluated,756,755,0,0,49,0.768212,1.146839,...,NaN,NaN,-38.468538,11.509663,0.272619,30.501116,18.630969,0.174542,-9.451594,-0.579104
2,trend_plain_contract,wf_2022,evaluated,253,253,0,0,49,0.833992,1.073453,...,NaN,NaN,-58.602980,12.185863,0.330955,36.419124,43.571257,0.280667,-40.195132,-1.551696
3,trend_plain_contract,wf_2023_2024,evaluated,756,755,15,0,49,0.754967,1.154651,...,NaN,NaN,-40.386846,10.785130,0.277116,41.557303,13.337480,0.189594,1.604592,0.206064
4,trend_plus_slope_low_cap15,wf_2022,evaluated,253,253,0,64,77,0.845850,1.059119,...,0.984375,0.896104,-61.084253,8.018977,0.229565,59.461195,20.450912,0.188882,-17.153062,-0.732730
5,trend_plus_slope_low_cap15,wf_2023_2024,evaluated,756,755,15,188,220,0.686093,1.219710,...,0.851064,0.745455,-45.161504,16.493124,0.320899,32.530872,19.911251,0.220942,-7.421838,-0.318639
6,trend_plus_slope_low_cap15_weighted,wf_2022,evaluated,253,253,0,64,77,0.810277,1.091642,...,0.906250,0.831169,-58.493302,9.398367,0.236715,61.152506,18.220379,0.177434,-15.461751,-0.751948
7,trend_plus_slope_low_cap15_weighted,wf_2023_2024,evaluated,756,755,15,188,220,0.737748,1.170573,...,0.904255,0.863636,-50.663499,12.535982,0.270040,40.445822,12.470085,0.183753,0.493111,0.127238
8,trend_plus_trend_break_any_cap10,wf_2022,evaluated,253,253,0,144,144,0.869565,1.031602,...,0.986111,0.986111,-62.434936,10.178761,0.234031,68.105462,18.167993,0.136776,-8.508794,-0.369612
9,trend_plus_trend_break_any_cap10,wf_2023_2024,evaluated,756,755,8,400,410,0.773510,1.132117,...,0.922500,0.902439,-59.204964,12.679734,0.173082,38.433877,10.947237,0.143390,-1.518833,-0.062113


,fold,candidate_relative_return,candidate_max_dd,candidate_avg_pos,baseline_relative_return,baseline_max_dd,baseline_avg_pos,relative_delta_vs_baseline,max_dd_delta_vs_baseline,candidate_gate,...,valid_relative_return,valid_max_dd,valid_avg_pos,trend_target_samples,sequence_risk_samples,drawdown_control_samples,after_accuracy,slice_failed_checks,failed_checks_json,gate_loss
0,wf_2022,36.419124,43.571257,0.280667,36.419124,43.571257,0.280667,0.000000,0.000000,False,...,-58.602980,12.185863,0.330955,0,0,49,0.833992,[],"[""candidate_max_dd""]",1.904750
1,wf_2023_2024,30.501116,18.630969,0.174542,30.501116,18.630969,0.174542,0.000000,0.000000,False,...,-38.468538,11.509663,0.272619,0,0,49,0.768212,[],"[""candidate_max_dd""]",0.242065
2,wf_2022,36.419124,43.571257,0.280667,36.419124,43.571257,0.280667,0.000000,0.000000,False,...,-58.602980,12.185863,0.330955,0,0,49,0.833992,[],"[""candidate_max_dd""]",1.904750
3,wf_2023_2024,41.557303,13.337480,0.189594,30.501116,18.630969,0.174542,11.056187,-5.293489,True,...,-40.386846,10.785130,0.277116,15,0,49,0.754967,[],[],0.000000
4,wf_2022,59.461195,20.450912,0.188882,36.419124,43.571257,0.280667,23.042070,-23.120345,False,...,-61.084253,8.018977,0.229565,0,64,77,0.845850,[],"[""candidate_max_dd""]",0.363394
5,wf_2023_2024,32.530872,19.911251,0.220942,30.501116,18.630969,0.174542,2.029756,1.280282,False,...,-45.161504,16.493124,0.320899,15,188,220,0.686093,"[""all_accuracy""]","[""candidate_max_dd"", ""slice_gate""]",1.327417
6,wf_2022,61.152506,18.220379,0.177434,36.419124,43.571257,0.280667,24.733381,-25.350877,False,...,-58.493302,9.398367,0.236715,0,64,77,0.810277,[],"[""candidate_max_dd""]",0.214692
7,wf_2023_2024,40.445822,12.470085,0.183753,30.501116,18.630969,0.174542,9.944706,-6.160884,True,...,-50.663499,12.535982,0.270040,15,188,220,0.737748,[],[],0.000000
8,wf_2022,68.105462,18.167993,0.136776,36.419124,43.571257,0.280667,31.686338,-25.403264,False,...,-62.434936,10.178761,0.234031,0,144,144,0.869565,[],"[""candidate_max_dd""]",0.211200
9,wf_2023_2024,38.433877,10.947237,0.143390,30.501116,18.630969,0.174542,7.932761,-7.683732,True,...,-59.204964,12.679734,0.173082,8,400,410,0.773510,"[""trend_slice_accuracy""]","[""slice_gate""]",1.000000


,candidate,decision,fold_gate_pass_count,candidate_gate_pass_count,baseline_gate_pass_count,drift_gate_pass_count,slice_gate_pass_count,failed_folds_json,gate_loss,min_relative_return,max_max_dd,mean_relative_delta_vs_baseline,max_dd_delta_vs_baseline,promotion_allowed,next_mechanism
0,trend_plus_slope_low_cap15_weighted,bc_walkforward_drift_mixed_no_promotion,1,1,0,2,2,"[""wf_2022""]",0.214692,40.445822,18.220379,17.339043,-6.160884,False,inspect_failed_fold_slices_or_checkpoint_selec...
1,trend_plain_contract,bc_walkforward_drift_mixed_no_promotion,1,1,0,2,2,"[""wf_2022""]",1.904750,36.419124,43.571257,5.528093,0.000000,False,inspect_failed_fold_slices_or_checkpoint_selec...
2,trend_plus_trend_break_any_cap10,bc_walkforward_drift_failed_no_promotion,0,1,0,2,1,"[""wf_2022"", ""wf_2023_2024""]",1.211200,38.433877,18.167993,19.809550,-7.683732,False,block_ppo_until_drawdown_drift_or_slice_fit_is...
3,trend_plus_slope_low_cap15,bc_walkforward_candidate_gate_failed_no_promotion,0,0,0,2,1,"[""wf_2022"", ""wf_2023_2024""]",1.690811,32.530872,20.450912,12.535913,1.280282,False,revise_training_targets_before_ppo
4,base_plain_contract,bc_walkforward_candidate_gate_failed_no_promotion,0,0,0,2,2,"[""wf_2022"", ""wf_2023_2024""]",2.146815,30.501116,43.571257,0.000000,0.000000,False,revise_training_targets_before_ppo


{
  "experiment": "sequence_trend_joint_bc_screen_v1",
  "created_at": "2026-05-26T06:54:27.849306+00:00",
  "decision": "sequence_trend_joint_bc_mixed_no_promotion",
  "selected_candidate": "trend_plus_slope_low_cap15_weighted",
  "selected_summary": {
    "candidate": "trend_plus_slope_low_cap15_weighted",
    "decision": "bc_walkforward_drift_mixed_no_promotion",
    "fold_gate_pass_count": 1,
    "candidate_gate_pass_count": 1,
    "baseline_gate_pass_count": 0,
    "drift_gate_pass_count": 2,
    "slice_gate_pass_count": 2,
    "failed_folds_json": "[\"wf_2022\"]",
    "gate_loss": 0.21469195670748295,
    "min_relative_return": 40.44582151415278,
    "max_max_dd": 18.220379350612244,
    "mean_relative_delta_vs_baseline": 17.33904344016525,
    "max_dd_delta_vs_baseline": -6.160883942381377,
    "promotion_allowed": false,
    "next_mechanism": "inspect_failed_fold_slices_or_checkpoint_selection_before_ppo"
  },
  "selected_walkforward_summary": {
    "decision": "bc_walkforward_

In [2]:

import json
from IPython.display import Markdown, display
selection = json.loads((CARD / "selection.json").read_text(encoding="utf-8"))
summary = json.loads((CARD / "summary.json").read_text(encoding="utf-8"))
conclusion_lines = "\n".join(f"- {line}" for line in summary["conclusion"])
selected = selection.get("selected_summary", {})
summary_md = "\n".join([
    "## Runtime Conclusion",
    "",
    f"Decision: `{selection['decision']}`. Promotion is blocked.",
    "",
    f"Selected candidate: `{selection['selected_candidate']}`.",
    "",
    f"Selected aggregate: fold gates `{selected.get('fold_gate_pass_count')}`, candidate gates `{selected.get('candidate_gate_pass_count')}`, drift gates `{selected.get('drift_gate_pass_count')}`, min relative `{selected.get('min_relative_return')}`, max MaxDD `{selected.get('max_max_dd')}`.",
    "",
    conclusion_lines,
    "",
    f"Next action: `{selection['next_mechanism']}`.",
])
display(Markdown(summary_md))
candidate_screen


## Runtime Conclusion

Decision: `sequence_trend_joint_bc_mixed_no_promotion`. Promotion is blocked.

Selected candidate: `trend_plus_slope_low_cap15_weighted`.

Selected aggregate: fold gates `1`, candidate gates `1`, drift gates `2`, min relative `40.44582151415278`, max MaxDD `18.220379350612244`.

- wf_2022: trend_plus_slope_low_cap15_weighted relative=61.15pp, MaxDD=18.22%, delta vs base rel=24.73pp, delta MaxDD=-25.35pp, slice_gate=True, fold_gate=False.
- wf_2023_2024: trend_plus_slope_low_cap15_weighted relative=40.45pp, MaxDD=12.47%, delta vs base rel=9.94pp, delta MaxDD=-6.16pp, slice_gate=True, fold_gate=True.
- No runtime promotion: this card evaluates target composition and BC warm-start behavior only and skips wf_2020 by leakage guard.

Next action: `inspect_failed_fold_action_distribution_then_try_drawdown_drift_reward_or_checkpoint_selector`.

,candidate,decision,fold_gate_pass_count,candidate_gate_pass_count,baseline_gate_pass_count,drift_gate_pass_count,slice_gate_pass_count,failed_folds_json,gate_loss,min_relative_return,max_max_dd,mean_relative_delta_vs_baseline,max_dd_delta_vs_baseline,promotion_allowed,next_mechanism
0,trend_plus_slope_low_cap15_weighted,bc_walkforward_drift_mixed_no_promotion,1,1,0,2,2,"[""wf_2022""]",0.214692,40.445822,18.220379,17.339043,-6.160884,False,inspect_failed_fold_slices_or_checkpoint_selec...
1,trend_plain_contract,bc_walkforward_drift_mixed_no_promotion,1,1,0,2,2,"[""wf_2022""]",1.904750,36.419124,43.571257,5.528093,0.000000,False,inspect_failed_fold_slices_or_checkpoint_selec...
2,trend_plus_trend_break_any_cap10,bc_walkforward_drift_failed_no_promotion,0,1,0,2,1,"[""wf_2022"", ""wf_2023_2024""]",1.211200,38.433877,18.167993,19.809550,-7.683732,False,block_ppo_until_drawdown_drift_or_slice_fit_is...
3,trend_plus_slope_low_cap15,bc_walkforward_candidate_gate_failed_no_promotion,0,0,0,2,1,"[""wf_2022"", ""wf_2023_2024""]",1.690811,32.530872,20.450912,12.535913,1.280282,False,revise_training_targets_before_ppo
4,base_plain_contract,bc_walkforward_candidate_gate_failed_no_promotion,0,0,0,2,2,"[""wf_2022"", ""wf_2023_2024""]",2.146815,30.501116,43.571257,0.000000,0.000000,False,revise_training_targets_before_ppo
